# Matched Train10K crop ablation — C / P3 / R7

Default `MODEL_FILTER="P3"` runs P3 at **1.5 and 2.0**. For three parallel Kaggle sessions use `"C"`, `"P3"`, `"R7"`. Optional `CROP_FILTER=[1.5]` or `[2.0]` runs one crop independently.

Attach the existing pooled 00d resources (including frozen Train10K/Val3K), CelebA/LCC images and MiniFASNet source/checkpoint. Edit the paths below. Every run trains, calibrates Val3K and evaluates LCC at its **same crop**. No 2.7 training, full CelebA Test, detector or alignment.

The original micro SGD/batch256/max20/milestones[6,14]/early-stop8/patience4 recipe and C/P3/R7 losses are preserved. Crop geometry uses the original helper including its boundary behavior. Progress is printed for setup, batch 1/every 10 batches, Val and LCC.

Historical 2.7 CSV paths are optional and read only after new-run evaluation. Completed runs receive their own artifacts, ZIP and family summary even if the second crop is unfinished. Resume is independent per run; keep its full family/run directory in `OUTPUT_ROOT` when resuming.

Primary endpoint: LCC Combined AUC. The context-reliance explanation is a hypothesis. No final crop or method winner is selected automatically.


In [ ]:
# Edit paths here. Both notebooks use the same frozen 00d resource bundle.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/micro_crop_train_10k_v1'
MODEL_FILTER='P3'
CROP_FILTER=None  # None, [1.5], [2.0], or [1.5, 2.0].
HISTORICAL_CROP27_CSV=None
HISTORICAL_P3_R7_CSV=None
LOG_EVERY_BATCHES=10
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
EARLIEST_EARLY_STOP_EPOCH=8
EARLY_STOP_PATIENCE=4
WARMUP_EPOCHS=1
CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=2.7
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12

RUNS={
 'C_crop15':{'family':'C','method':'CLEAN','crop_factor':1.5,'clean_ce_weight':1.,'spectral_ce_weight':0.},
 'C_crop20':{'family':'C','method':'CLEAN','crop_factor':2.0,'clean_ce_weight':1.,'spectral_ce_weight':0.},
 'P3_crop15':{'family':'P3','method':'WORST','crop_factor':1.5,'clean_ce_weight':.75,'spectral_ce_weight':.25},
 'P3_crop20':{'family':'P3','method':'WORST','crop_factor':2.0,'clean_ce_weight':.75,'spectral_ce_weight':.25},
 'R7_crop15':{'family':'R7','method':'HARMFUL','crop_factor':1.5,'clean_ce_weight':.75,'spectral_ce_weight':.25},
 'R7_crop20':{'family':'R7','method':'HARMFUL','crop_factor':2.0,'clean_ce_weight':.75,'spectral_ce_weight':.25},
}
def select_runs(model_filter,crop_filter):
    if not isinstance(model_filter,str) or model_filter not in ('C','P3','R7'):raise ValueError('MODEL_FILTER must be C, P3 or R7')
    if crop_filter is not None and (not isinstance(crop_filter,(list,tuple)) or not crop_filter or len(set(crop_filter))!=len(crop_filter) or any(isinstance(c,bool) or c not in (1.5,2.0) for c in crop_filter)):
        raise ValueError('CROP_FILTER must be None or a unique non-empty list of 1.5 / 2.0')
    return [name for name,spec in RUNS.items() if spec['family']==model_filter and (crop_filter is None or spec['crop_factor'] in crop_filter)]


In [ ]:
PROTOCOL_TEXT="# Matched Train10K crop ablation: C / P3 / R7, v1\n\n## Question and hypothesis\n\nDoes tighter source-training context improve transfer to LCC when training, source calibration and target evaluation use the same crop? Excessive expansion may encourage reliance on source-specific surrounding cues unavailable in already tightly cropped target imagery. This is a hypothesis, not a measured conclusion. Prior inference-only observations do not establish matched-training performance.\n\n## Design and fixed controls\n\nSix possible runs: C_crop15, C_crop20, P3_crop15, P3_crop20, R7_crop15, R7_crop20. The only treatment within each mechanism is crop factor 1.5 or 2.0. Do not retrain 2.7x. MODEL_FILTER is one of C/P3/R7, default P3. CROP_FILTER=None selects both crops; [1.5] or [2.0] selects one. Each family/run is independent and has no writable dependency on another family.\n\nRead the existing mini_train10k.csv and mini_val3k.csv, seed 43; no resampling. Train seed 100. Reuse notebooks 06/07's MiniFASNetV2 binary/11/5/40 heads, official PAD-pretrained feature initialization, spectral/auxiliary logic, dataset order, augmentation streams and metrics. Record identical initial-state SHA across new runs. Bbox jitter is per sample/epoch with its own seeded RNG; base augmentation and spectral gains retain the micro streams, preventing crop or spectral selection from changing the underlying augmentation draw policy.\n\nKeep frozen valid raw SCRFD boxes and eligibility, jitter p=.20, scale .95–1.05 and center shift ±.05 face size. Apply the run's crop to jittered training boxes and unjittered evaluation boxes. Use the exact current CropImage-compatible geometry including boundary scale limit and inward translation, 80x80 BGR float32 divided by 255. Keep HorizontalFlip .5, ISONoise .2, brightness/contrast limits .2 with probability .3, MotionBlur limit 5 with probability .2. No normalization, mixup, cutmix, weighted sampler, smoothing, alignment, detector or GT fallback.\n\n## Training and mechanism losses\n\nSGD lr .005, momentum .9, weight decay 5e-4; batch 256; maximum 20 epochs; MultiStepLR [6,14], gamma .2, stepped after each epoch. Preserve earliest stopping epoch 8 and patience 4 with minimum Val3K ACER/higher AUC tie-break. No phase-aware stopping change. C is eligible from epoch 1. P3/R7 have one clean warm-up epoch excluded from best selection and patience, then spectral-active selection.\n\n```\nL_aux = .1 L_spoof_type + .1 L_lighting + L_attributes\nL_C = CE_clean + L_aux\nL_P3 = .75 CE_clean + .25 CE_worst + L_aux\nL_R7 = .75 CE_clean + .25 CE_harmful + L_aux\n```\n\nAuxiliary loss is clean only; attributes are Real only. P3/R7 warm-up uses CE_clean+L_aux. C does not generate spectral views. Spectral bands LOW/MID/HIGH have centers [.15,.45,.75], sigma .10, gain Uniform(.65,.90), preserved DC/phase, no amplification. Selection reuses tested eval/no_grad forwards with mode restored, selecting the lowest label-aligned margin. The existing micro helper also gathers clean selection margin without BN updates; only R7 uses it to gate the loss. R7 detaches harmful=(m_worst<m_clean), normalizes selected-view CE over harmful samples and uses differentiable zero for no harmful samples. No KL, SmoothL1, margin regression, random/all-band supervision.\n\n## Matched evaluation and diagnostics\n\nOnly Val3K is inferred within the epoch loop, at that run's training crop. Reload best, recalibrate the source minimum-ACER threshold and freeze it before full LCC inference, always at the same crop. No CelebA full Test is loaded or evaluated. LCC train/dev/eval raw counts are 8299/2948/7580; combined is 18827. Concatenate physical prediction tables and recompute pooled metrics. Never tune on LCC or average split metrics.\n\nPrimary endpoint is LCC Combined AUC, with official-evaluation AUC and pooled EER/HTER/TPR@FPR1% alongside Val3K ACER/AUC as source guard. Save AUC/EER/TPR, APCER/BPCER/ACER/HTER/accuracy, locked threshold, candidate/scored counts and coverage. Failed localization candidates stay unscored; image errors stop rather than silently alter eligibility.\n\nCollect actual crop geometry without extra decoding or RNG draws: boundary-limited fraction plus mean/median crop-width/image-width and crop-height/image-height. Training diagnostics describe each epoch's jittered boxes; Val and LCC describe deterministic boxes. Pooled geometry concatenates physical scored samples. These diagnostics do not influence checkpoint selection.\n\n## Resume, artifacts and historical comparison\n\nEach run has config, best.pth, last_state.pth, history, spectral diagnostics (also produced for C), crop diagnostics, freeze, Val3K/LCC metrics and predictions. Resume stores model/optimizer/scheduler/scaler/epoch, committed best checkpoint payload, ACER/AUC/patience, histories/geometry summaries and RNG states. Only one correctly matched restore_rng definition exists; run/config/crop checks reject cross-run resumes. Restore the committed best payload before continuing.\n\nOutput root is /kaggle/working/micro_crop_train_10k_v1/<family>/<run>/. Every completed run produces complete standalone artifacts and a run ZIP; each family execution creates micro_crop_<family>_10k_v1.zip with protocol and family comparison/report. Export summaries after each run so another unfinished crop does not block completed artifacts.\n\nHISTORICAL_CROP27_CSV and HISTORICAL_P3_R7_CSV are optional read-only result paths, loaded after new-run evaluation. Accept historical C/R0_clean, P3/R1_p3_ce25, R7/R7_harmful_gated_worst. Append a historical 2.7 row and deltas only when measured values are available; missing controls remain pending. Training does not depend on these paths. Do not infer missing metrics or tune a crop from target results.\n\n## Interpretation and validation\n\nAsk explicitly whether reduced training context improves cross-domain separability rather than only changing calibration. Higher LCC AUC is stronger descriptive evidence of transfer improvement; HTER-only improvement may reflect operating-point effects. Look for consistent effects across C/P3/R7 after merging independent family tables; larger spectral-only effects may indicate mechanism interaction. Single-seed 10K screening and historical comparisons do not prove the hypothesis or establish a final method/crop winner.\n\nValidate parse/compile/cleared outputs, exact filters/runs/crops, original helpers/streams, matched crop paths, three loss formulas, detached zero-safe harmful gate, BN-safe selection, unique resume restoration, no Test/detector/GT/target fitting, and pooled metrics using tiny offline checks. Do not launch expensive training during notebook creation.\n"

import hashlib, time, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

AUG_SEED=TRAIN_SEED+300000
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
selected=select_runs(MODEL_FILTER,CROP_FILTER)
print('[Setup] Selected runs:',selected,flush=True)
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen micro recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',
             root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()

def geometry_summary(g):
    return {'number_of_evaluated_samples':len(g),'boundary_clipped_fraction':float(g[:,0].mean()),
      'mean_crop_width_over_image_width':float(g[:,1].mean()),'mean_crop_height_over_image_height':float(g[:,2].mean()),
      'median_crop_width_over_image_width':float(np.median(g[:,1])),'median_crop_height_over_image_height':float(np.median(g[:,2]))}

def crop_geometry(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    requested_w,requested_h=box_w*float(scale),box_h*float(scale)
    requested_cx,requested_cy=(x1+x2)/2,(y1+y2)/2
    limited=(requested_cx-requested_w/2<0 or requested_cy-requested_h/2<0 or requested_cx+requested_w/2>src_w-1 or requested_cy+requested_h/2>src_h-1)
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return [float(limited or used<float(scale)),(rx-lx+1)/src_w,(by-ty+1)/src_h]

def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(CSMR_BAND_CENTERS,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*CSMR_BAND_SIGMA**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def bbox_jitter(rec,rng):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if rng.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=rng.uniform(.95,1.05)
        cx+=rng.uniform(-.05,.05)*base;cy+=rng.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]

def copied_aug_bgr_seeded(patch,seed):
    # Augmentation RNG cannot consume bbox or spectral RNG state.
    random.seed(seed);np.random.seed(seed)
    if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(seed)
    return copied_aug_bgr(patch)

def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('mini_train10k');val=load_frame('mini_val3k')
if (len(train),len(val))!=(10000,3000):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if 'sampling_seed' in frame and set(frame.sampling_seed)!={DATA_SEED}:raise RuntimeError('Frozen micro seed differs')
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen mini split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen micro batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/MODEL_FILTER).mkdir(exist_ok=True)
(out/'MICRO_CROP_TRAIN_C_P3_R7_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()},flush=True)

def init_model():
    seed_all(TRAIN_SEED)
    model=PADNet(True)
    raw=torch.load(pretrained,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Official PAD checkpoint format differs')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):
        raise RuntimeError('Official PAD checkpoint architecture differs')
    compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
    target=model.trunk.state_dict()
    if set(compatible)!=set(target) or any(tuple(compatible[k].shape)!=tuple(target[k].shape) for k in compatible):
        raise RuntimeError('PAD pretrained trunk keys differ')
    model.trunk.load_state_dict(compatible,strict=True)
    return model.to(device)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,epoch=0,crop_factor=2.7):
        self.rows=frame.reset_index(drop=True);self.training=training;self.epoch=epoch;self.crop_factor=crop_factor
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        box=bbox_jitter(rec,random.Random(BBOX_SEED+self.epoch*100000+i)) if self.training else rec['bbox_xyxy']
        patch=mini_crop_bgr(image,box,scale=self.crop_factor)
        geometry=np.asarray(crop_geometry(image,box,scale=self.crop_factor),dtype=np.float64)
        if self.training:patch=copied_aug_bgr_seeded(patch,AUG_SEED+self.epoch*100000+i)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key,geometry

class LCCData(Dataset):
    def __init__(self,split,crop_factor):
        self.split=split;self.crop_factor=crop_factor;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'],scale=self.crop_factor)),int(row.label),key,np.asarray(crop_geometry(image,rec['bbox_xyxy'],scale=self.crop_factor),dtype=np.float64)

def loader(ds,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,
                      drop_last=False,generator=gen)

def infer(model,dl,lcc=False,label='Evaluation'):
    model.eval();keys=[];ys=[];zs=[];geometry=[];started=time.monotonic()
    print('['+label+'] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc:x,y,k,g=batch
            else:x,y,_,_,_,_,k,g=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy());geometry.append(g.numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):
                print(f'[{label}] batch {step}/{len(dl)}, elapsed {time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs),np.concatenate(geometry)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))


def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active):
    method=spec['method'];n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_worst=None
    spectral_active=active and method!='CLEAN'
    if spectral_active:
        selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;harmful_count=0;spectral_logits=None
        if spectral_active:
            spectral_logits=model(selected);ce_each=F.cross_entropy(spectral_logits,binary,reduction='none')
            if method=='HARMFUL':
                harmful=(select_worst<select_clean).detach();harmful_count=int(harmful.sum())
                spectral_ce=(ce_each*harmful).sum()/harmful_count if harmful_count else spectral_logits.sum()*0
            else:spectral_ce=ce_each.mean()
        loss=.75*clean_ce+.25*spectral_ce+aux_total if spectral_active else clean_ce+aux_total
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary).mean()
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
          'spectral_ce':float(spectral_ce.detach()),'aux_loss':float(aux_total.detach()),
          'clean_aligned_margin':float(cm),'selection_clean_margin':float(select_clean.mean()) if select_clean is not None else 0.,
          'worst_margin':float(select_worst.mean()) if select_worst is not None else 0.,
          'margin_drop':float((select_clean-select_worst).mean()) if select_worst is not None else 0.,
          'spectral_flip_rate':flips/n,'harmful_count':harmful_count,'harmful_fraction':harmful_count/n,
          'harmful_ce':float(spectral_ce.detach()) if method=='HARMFUL' and spectral_active else 0.,
          'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts

def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
      'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state()}

def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain'])

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'mini_train10k.csv','val_manifest':'mini_val3k.csv',
      'data_seed':DATA_SEED,'train_seed':TRAIN_SEED,'train_crop_factor':spec['crop_factor'],'eval_crop_factor':spec['crop_factor'],
      'base_order':'torch Generator(TRAIN_SEED+epoch)','bbox_seed':BBOX_SEED,'augmentation_seed':AUG_SEED,'gain_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,'num_workers':NUM_WORKERS,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':0 if spec['method']=='CLEAN' else WARMUP_EPOCHS,
      'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'spectral':{'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,'gain':[.65,.90],'preserve_dc':CSMR_PRESERVE_DC},
      'input':'80x80 BGR float [0,1]','augmentation_api':AUGMENTATION_API,'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked,crop_factor,crop_diagnostics):
    scored_predictions=[];all_geometry=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z,g=infer(model,loader(LCCData(split,crop_factor)),True,label=folder.name+' LCC '+split)
        crop_diagnostics['lcc_'+split]=geometry_summary(g);all_geometry.append(g)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)
    crop_diagnostics['lcc_combined']=geometry_summary(np.concatenate(all_geometry))


def run_one(name,spec):
    folder=out/spec['family']/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True)
    print('['+name+'] Starting matched crop',spec['crop_factor'],flush=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/(k+'_metrics.json') for k in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping',flush=True)
        return
    model=init_model()
    initial_sha=state_sha(model.state_dict())
    write_json(folder/'initialization_audit.json',{'initial_state_sha256':initial_sha,'train_seed':TRAIN_SEED,'pretrained_checkpoint':str(pretrained)})
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None;crop_diagnostics={'training_epochs':{}}
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state['initial_state_sha256']!=initial_sha:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint'];crop_diagnostics=state['crop_diagnostics']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume committed best differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start,flush=True)
    val_dl=loader(CelebaData(val,crop_factor=spec['crop_factor']))
    masks=radial_masks() if spec['method']!='CLEAN' else None
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            warmup=spec['method']!='CLEAN' and epoch<=WARMUP_EPOCHS
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;harmful_count_total=0;train_geometry=[]
            train_ds=CelebaData(train,training=True,epoch=epoch,crop_factor=spec['crop_factor'])
            train_dl=loader(train_ds,shuffle=True,epoch=epoch);started=time.monotonic()
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, phase={"warmup" if warmup else "active"}, lr={optimizer.param_groups[0]["lr"]:g}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_,g) in enumerate(train_dl,1):
                train_geometry.append(g.numpy())
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                harmful_count_total+=scalars['harmful_count']
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-started:.1f}s',flush=True)
            crop_diagnostics['training_epochs'][str(epoch)]=geometry_summary(np.concatenate(train_geometry))
            keys,y,z,g=infer(model,val_dl,label=f'{name} epoch {epoch} Val3K')
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                else:no_improve+=1
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'rng':snapshot_rng(gain_rng),'initial_state_sha256':initial_sha,
                'best_checkpoint':best_checkpoint,'crop_diagnostics':crop_diagnostics,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            write_json(folder/'crop_diagnostics.json',crop_diagnostics)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<= (0 if spec['method']=='CLEAN' else WARMUP_EPOCHS):
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z,g=infer(model,val_dl,label=name+' frozen Val3K')
    crop_diagnostics['val3k']=geometry_summary(g)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'val3k_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'val3k_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val3k_metrics':vm,
      'train_crop_factor':spec['crop_factor'],'eval_crop_factor':spec['crop_factor'],'threshold_source':'frozen Val3K at matched crop'}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_lcc(model,folder,locked,spec['crop_factor'],crop_diagnostics)
    write_json(folder/'crop_diagnostics.json',crop_diagnostics)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'crop_factor':spec['crop_factor'],'initial_state_sha256':initial_sha})
    del model
    torch.cuda.empty_cache()


In [ ]:
COMPARISON_KEYS=('val3k_acer','val3k_auc','lcc_evaluation_auc','lcc_evaluation_hter','lcc_combined_auc',
 'lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')

def historical_row(family):
    aliases={'C':('C','R0_clean'),'P3':('P3','R1_p3_ce25'),'R7':('R7','R7_harmful_gated_worst')}[family]
    notes=[]
    for value in (HISTORICAL_CROP27_CSV,HISTORICAL_P3_R7_CSV):
        if value is None:continue
        path=Path(value)
        if not path.is_file():notes.append('Historical path unavailable: '+str(path));continue
        frame=pd.read_csv(path,keep_default_na=False)
        idcol=next((c for c in ('run_id','model','method') if c in frame),None)
        if idcol is None:notes.append('No model/run ID column in '+str(path));continue
        matches=frame.loc[frame[idcol].astype(str).isin(aliases)]
        if matches.empty:continue
        if len(matches)!=1:notes.append('Ambiguous historical rows for '+family+' in '+str(path)+'; skipped.');continue
        raw=matches.iloc[0].to_dict();row={'model':family,'run_id':str(raw[idcol]),'crop_factor':2.7,'status':'historical','source_csv':str(path)}
        for key,value in raw.items():
            target='val3k_'+key[4:] if key.startswith('val_') else key
            if target.startswith(('val3k_','lcc_')):
                try:v=float(value)
                except (TypeError,ValueError):continue
                if np.isfinite(v):row[target]=v
            elif key=='selected_epoch':row[key]=value
        if not any(k in row for k in COMPARISON_KEYS):notes.append('No comparable historical metrics in '+str(path));continue
        return row,notes
    return None,notes

def comparison_table(rows):
    lines=['| Model | Crop | Status | '+' | '.join(COMPARISON_KEYS)+' |','|---|---:|---|'+'---:|'*len(COMPARISON_KEYS)]
    for row in rows:
        lines.append('| '+row['model']+' | '+str(row['crop_factor'])+' | '+row['status']+' | '+' | '.join(
          f'{row[k]:.6f}' if isinstance(row.get(k),(int,float)) else 'pending' for k in COMPARISON_KEYS)+' |')
    return lines

def export_comparison(family):
    folder=out/family;rows=[]
    for name,spec in RUNS.items():
        if spec['family']!=family:continue
        run_folder=folder/name
        if not (run_folder/'completion.json').is_file():continue
        done=read_json(run_folder/'completion.json')
        row={'model':family,'run_id':name,'crop_factor':spec['crop_factor'],'status':'complete','selected_epoch':done['best_epoch']}
        for split in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
            for key,value in read_json(run_folder/(split+'_metrics.json')).items():row[split+'_'+key]=value
        rows.append(row)
    historical,notes=historical_row(family)
    if historical:
        for row in rows:
            for key in COMPARISON_KEYS:
                row['delta_vs_historical27_'+key]=row[key]-historical[key] if key in historical else None
        rows.append(historical)
    pd.DataFrame(rows).to_csv(folder/'crop_train_comparison.csv',index=False)
    write_json(folder/'crop_train_comparison.json',rows)
    report=['# '+family+' matched train/evaluation crop study','',
      'Train10K / Val3K unchanged; seed 100, maximum 20 epochs, milestones [6,14], original earliest-stop epoch 8 and patience 4.',
      'Train, Val3K threshold calibration and LCC all use the same run-specific crop. No CelebA full Test.',
      'Primary endpoint: LCC Combined AUC. Val3K ACER/AUC are the source-domain guard.','',*comparison_table(rows),'',
      '## Does reducing training context improve separability rather than only calibration?','']
    completed=[r for r in rows if r['status']=='complete']
    if historical:
        for row in completed:
            da=row.get('delta_vs_historical27_lcc_combined_auc');dh=row.get('delta_vs_historical27_lcc_combined_hter')
            de=row.get('delta_vs_historical27_lcc_evaluation_auc');dv=row.get('delta_vs_historical27_val3k_acer')
            text='- '+row['run_id']+': '
            if da is None:text+='historical pooled AUC missing; separability comparison pending.'
            elif da>0:text+=f'pooled AUC delta {da:+.6f}; descriptively supports transfer/separability improvement, subject to source guard and single-seed limitations.'
            elif dh is not None and dh<0:text+=f'HTER improves ({dh:+.6f}) without pooled AUC improvement ({da:+.6f}); an operating-point/calibration effect is more plausible than improved separability.'
            else:text+=f'pooled AUC delta {da:+.6f}; no pooled AUC improvement observed versus this historical control.'
            if de is not None:text+=f' Official-evaluation AUC delta {de:+.6f}.'
            if dv is not None:text+=f' Val3K ACER delta {dv:+.6f} (positive is worse).'
            report.append(text)
        report+=['','## Deltas versus historical 2.7x','',
          '| Run | '+' | '.join(COMPARISON_KEYS)+' |','|---|'+'---:|'*len(COMPARISON_KEYS)]
        for row in completed:
            report.append('| '+row['run_id']+' | '+' | '.join(f'{row["delta_vs_historical27_"+k]:+.6f}' if row.get('delta_vs_historical27_'+k) is not None else 'pending' for k in COMPARISON_KEYS)+' |')
        for row in completed:
            for split in ('lcc_evaluation','lcc_combined'):
                for suffix in ('candidate_n','scored_n'):
                    key=split+'_'+suffix
                    if key in historical and row[key]!=historical[key]:notes.append(row['run_id']+' '+key+' differs from historical control; interpret comparability cautiously.')
    else:
        report.append('Historical 2.7x control unavailable: matched 1.5/2.0 results are measured, but improvement versus 2.7x remains pending.')
    if len(completed)==2:
        a=next(r for r in completed if r['crop_factor']==1.5);b=next(r for r in completed if r['crop_factor']==2.)
        report.append(f'Within {family}, 1.5 minus 2.0: pooled AUC {a["lcc_combined_auc"]-b["lcc_combined_auc"]:+.6f}; pooled HTER {a["lcc_combined_hter"]-b["lcc_combined_hter"]:+.6f}.')
    report+=['','## Shared crop effect or spectral interaction?','',
      'This execution contains one independent family. Merge the C/P3/R7 crop_train_comparison.csv files after their sessions finish.',
      'Same-direction AUC changes across all three families support a common crop effect. Larger or distinct P3/R7 changes may indicate a spectral interaction; no cross-family conclusion is inferred from one family.',
      'Crop geometry summaries include training by epoch, deterministic Val3K and LCC split/pooled mean/median dimensions and boundary-limited fraction.',
      'Excessive source context encouraging reliance on domain-specific cues remains a hypothesis. These single-seed 10K results do not prove the mechanism or select a final crop.',
      'Historical comparisons require matched frozen manifests/recipe; unavailable metrics remain pending. No automatic target-based model selection.',
      '',*notes]
    (folder/'crop_train_report.md').write_text('\n'.join(report)+'\n')
    return rows

def make_run_zip(name,spec):
    folder=out/spec['family']/name;path=out.parent/(name+'_10k_v1.zip')
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(folder.rglob('*')):
            if file.is_file():archive.write(file,(Path(spec['family'])/name/file.relative_to(folder)).as_posix())
        archive.write(out/'MICRO_CROP_TRAIN_C_P3_R7_PROTOCOL_v1.md','MICRO_CROP_TRAIN_C_P3_R7_PROTOCOL_v1.md')
    display(FileLink(str(path)))

def make_family_zip(family):
    folder=out/family;path=out.parent/f'micro_crop_{family}_10k_v1.zip'
    print('[Export] Writing family ZIP:',path,flush=True)
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(folder.rglob('*')):
            if file.is_file():archive.write(file,(Path(family)/file.relative_to(folder)).as_posix())
        archive.write(out/'MICRO_CROP_TRAIN_C_P3_R7_PROTOCOL_v1.md','MICRO_CROP_TRAIN_C_P3_R7_PROTOCOL_v1.md')
    display(FileLink(str(path)))

selected=select_runs(MODEL_FILTER,CROP_FILTER)
print('[Runner] Selected:',selected,flush=True)
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch 256 ran out of GPU memory; stop instead of changing the recipe') from exc
    export_comparison(MODEL_FILTER)
    make_run_zip(name,RUNS[name]);make_family_zip(MODEL_FILTER)
print('Finished selected matched-crop runs:',selected,flush=True)
